In [1]:
import pandas as pd
df=pd.read_csv("../data/APL_Logistics.csv", encoding="latin1")

In [ ]:
# Earlier we identified:

# Delivery Status
# Days for shipping (real)
# These should not be used for prediction.

In [2]:
df = df.drop(
    columns=[
        "Delivery Status",
        "Days for shipping (real)"
    ]
)

In [ ]:
# We don't know the actual shipping days yet.
# We don't know the final delivery status yet.
# Using them would make the model cheat

In [3]:
# Define Target Variable

y = df["Late_delivery_risk"]

In [4]:
# Define Features

X = df.drop("Late_delivery_risk", axis=1)

In [6]:
"""
Features (X)
↓
Predict
↓
Target (y)


X = Order information
y = Late_delivery_risk """


'\nFeatures (X)\n↓\nPredict\n↓\nTarget (y)\n\n\nX = Order information\ny = Late_delivery_risk '

In [7]:
# Check Feature Shape

print(X.shape)
print(y.shape)

(180519, 37)
(180519,)


In [8]:
# Check Missing Values Again

X.isnull().sum().sort_values(ascending=False).head(10)

Customer Lname                   8
Customer Zipcode                 3
Benefit per order                0
Days for shipment (scheduled)    0
Type                             0
Category Id                      0
Sales per customer               0
Customer Country                 0
Category Name                    0
Customer Fname                   0
dtype: int64

In [9]:
# Identify Numerical and Categorical Columns

num_cols = X.select_dtypes(include=['int64', 'float64']).columns

cat_cols = X.select_dtypes(include=['object', 'string']).columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

Numerical Columns: 19
Categorical Columns: 18


In [10]:
# Fill Missing Values

X["Customer Lname"] = X["Customer Lname"].fillna("Unknown")

X["Customer Zipcode"] = X["Customer Zipcode"].fillna(
    X["Customer Zipcode"].mode()[0]
)

In [11]:
# Verify Missing Values Removed

X.isnull().sum().sum()

np.int64(0)

In [12]:
# Identify Numerical and Categorical Features

num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols = X.select_dtypes(
    include=["object", "string"]
).columns

print("Numerical Columns:", len(num_cols))
print("Categorical Columns:", len(cat_cols))

Numerical Columns: 19
Categorical Columns: 18


In [13]:
# Inspect Categorical Features

cat_cols.tolist()


['Type',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Fname',
 'Customer Lname',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Department Name',
 'Market',
 'Order City',
 'Order Country',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Name',
 'Shipping Mode']

In [14]:
# Check High-Cardinality Columns

for col in cat_cols:
    print(col, ":", X[col].nunique())

Type : 4
Category Name : 50
Customer City : 563
Customer Country : 2
Customer Fname : 782
Customer Lname : 1110
Customer Segment : 3
Customer State : 46
Customer Street : 7458
Department Name : 11
Market : 5
Order City : 3597
Order Country : 164
Order Region : 23
Order State : 1089
Order Status : 9
Product Name : 118
Shipping Mode : 4


In [ ]:
# # Train-Test Split

# The dataset is divided into training and testing sets.

# - Training Data: Used to train the model
# - Testing Data: Used to evaluate model performance

# 80% Training
# 20% Testing

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (144415, 37)
Testing Features: (36104, 37)
Training Target: (144415,)
Testing Target: (36104,)


In [16]:
# # Identify Feature Types

# Machine learning models require numerical inputs.

# Before encoding categorical variables, we identify:

# - Numerical Features
# - Categorical Features

In [17]:
# Numerical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Categorical columns
cat_cols = X.select_dtypes(include=['object', 'string']).columns

print("Numerical Features:", len(num_cols))
print("Categorical Features:", len(cat_cols))

Numerical Features: 19
Categorical Features: 18


In [18]:
print("Numerical Columns:")
print(list(num_cols))

Numerical Columns:
['Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Category Id', 'Customer Id', 'Customer Zipcode', 'Department Id', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Price']


In [19]:
print("Categorical Columns:")
print(list(cat_cols))

Categorical Columns:
['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Fname', 'Customer Lname', 'Customer Segment', 'Customer State', 'Customer Street', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Shipping Mode']


In [20]:
# # Encoding Categorical Features

# Machine learning algorithms cannot process text values directly.

# Categorical variables are converted into numerical representations using One-Hot Encoding.

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            cat_cols
        )
    ],
    remainder='passthrough'
)

In [23]:
# # Apply Preprocessing

# The preprocessing pipeline is fitted on the training data and applied to both training and testing datasets.

In [24]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [25]:
print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (144415, 14911)
Processed Testing Shape: (36104, 14911)


In [26]:
print(type(X_train_processed))

<class 'scipy.sparse._csr.csr_matrix'>


In [27]:
# ## Sparse Matrix Verification

# After applying One-Hot Encoding, the transformed dataset is stored as a sparse matrix.

# Reason:
# - One-Hot Encoding creates thousands of binary columns.
# - Most values are zeros.
# - Sparse matrices store only non-zero values, reducing memory usage.

# Result:
# - Original Features: 37
# - Encoded Features: 14,911
# - Storage Type: scipy.sparse.csr_matrix